# Ingest Circuits CSV File

This notebook demonstrates a structured approach to ingesting circuits.csv:
* Read circuits.csv using the Spark DataFrame reader API
* Add metadata columns: source_file and ingestion_timestamp
* Write the results to the bronze Delta table

This version uses centralized configuration and helper functions for consistency.

In [0]:
%run ../00-common/01.Formula1_Configuration_Setup

In [0]:
# Check configured catalog
catalog_name

In [0]:
%run ../00-common/02.bronze-helper

In [0]:
# Check landing folder path
landing_folder_path

In [0]:
# Define source file and target table
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
# Preview source file path
source_file

In [0]:
# Read circuits CSV into a DataFrame
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
circuits_schema=StructType([StructField("circuitId",IntegerType()),
                                  StructField("url",StringType()),
                                  StructField("circuitName",StringType()),
                                  StructField("lat",DoubleType()),
                                  StructField("long",DoubleType()),
                                  StructField("locality",StringType()),
                                  StructField("country",StringType())
                                 ])
circuits_df=spark.read.format("csv").option("header","true").schema(circuits_schema).load(source_file)
display(circuits_df)

In [0]:
# Add ingestion metadata columns
circuits_final_df=add_ingestion_metadata(circuits_df)
display(circuits_final_df)

In [0]:
# Write circuits data to the bronze Delta table
(
    circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

In [0]:
# Preview target table name
table_name

In [0]:
%sql
-- Query the bronze circuits table
select * from formula1.bronze.circuits

In [0]:
# Display the bronze circuits table
display(spark.table(table_name))